# Task 2: MySQL Database Pipeline – TechReads Retail Analytics
**Module:** CMP-X304-0 Data Engineering  
**Database:** techreads_db  
**Table:** books  
**Input:** techreads_books.csv

## Step 1: Import Required Libraries

In [ ]:
# mysql-connector-python: official MySQL driver for Python
# allows us to connect to MySQL and execute SQL commands from Python
import mysql.connector

# pandas: used to read the CSV file and inspect data before insertion
import pandas as pd

## Step 2: Load and Inspect CSV Data

In [ ]:
# Load the CSV file produced by Task 1
df = pd.read_csv('techreads_books.csv', encoding='utf-8-sig')

# Preview the data to confirm it loaded correctly
print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

## Step 3: Connect to MySQL and Create Database

In [ ]:
# Establish connection to MySQL server
# Update 'password' to match your local MySQL root password
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='your_password_here'  # <-- change this
)
cursor = conn.cursor()

# Create the database if it does not already exist
cursor.execute('CREATE DATABASE IF NOT EXISTS techreads_db')
print('Database techreads_db created (or already exists).')

# Select the database for all subsequent operations
cursor.execute('USE techreads_db')
print('Using database: techreads_db')

## Step 4: Create the Books Table

In [ ]:
# Drop the table if it already exists to allow clean re-runs during testing
cursor.execute('DROP TABLE IF EXISTS books')

# Create the books table with appropriate data types and constraints
# - id: auto-incrementing primary key, uniquely identifies each record
# - title: VARCHAR(255) for variable-length book titles, NOT NULL as title is essential
# - price: DECIMAL(6,2) for accurate monetary values (up to 9999.99)
# - star_rating: TINYINT to store integer ratings 1-5, efficient for small numbers
# - availability: VARCHAR(50) for stock status text
# - category: VARCHAR(100) for book category classification
create_table_query = '''
CREATE TABLE books (
    id           INT AUTO_INCREMENT PRIMARY KEY,
    title        VARCHAR(255) NOT NULL,
    price        DECIMAL(6,2),
    star_rating  TINYINT,
    availability VARCHAR(50),
    category     VARCHAR(100)
)
'''
cursor.execute(create_table_query)
print('Table books created successfully.')

# Verify the table structure using DESCRIBE
cursor.execute('DESCRIBE books')
print('\nTable structure:')
for row in cursor.fetchall():
    print(row)

## Step 5: Import CSV Data into MySQL

In [ ]:
# SQL INSERT statement using parameterised queries
# %s placeholders prevent SQL injection and handle special characters in titles safely
insert_query = '''
    INSERT INTO books (title, price, star_rating, availability, category)
    VALUES (%s, %s, %s, %s, %s)
'''

# Iterate through each row in the DataFrame and insert into MySQL
inserted = 0
for _, row in df.iterrows():
    cursor.execute(insert_query, (
        row['title'],
        float(row['price']),
        int(row['star_rating']),
        row['availability'],
        row['category']
    ))
    inserted += 1

# Commit the transaction to permanently save all changes to the database
conn.commit()
print(f'Successfully inserted {inserted} records into books table.')

## Step 6: SQL Queries

In [ ]:
# Query 1: Select title, star_rating, price – sorted by price ascending
# Requirement: extract three columns and sort by one column (price)
query1 = '''
    SELECT title, star_rating, price
    FROM books
    ORDER BY price ASC
'''
cursor.execute(query1)
results1 = cursor.fetchall()

print('Query 1: All books ordered by price (low to high)')
print(f'{"Title":<65} {"Rating":<8} {"Price"}')
print('-' * 85)
for row in results1:
    print(f'{str(row[0]):<65} {str(row[1]):<8} £{row[2]}')

In [ ]:
# Query 2: Select title, price, availability – filter books with high ratings
# Identifies well-rated books, useful for publisher recommendation insights
query2 = '''
    SELECT title, price, availability
    FROM books
    WHERE star_rating >= 4
    ORDER BY star_rating DESC
'''
cursor.execute(query2)
results2 = cursor.fetchall()

print('Query 2: Books with star rating >= 4')
print(f'{"Title":<65} {"Price":<10} {"Availability"}')
print('-' * 90)
for row in results2:
    print(f'{str(row[0]):<65} £{str(row[1]):<9} {row[2]}')

In [ ]:
# Query 3: Select title, category, price – top 5 most expensive books
# Useful for pricing trend analysis across categories
query3 = '''
    SELECT title, category, price
    FROM books
    ORDER BY price DESC
    LIMIT 5
'''
cursor.execute(query3)
results3 = cursor.fetchall()

print('Query 3: Top 5 most expensive books')
print(f'{"Title":<65} {"Category":<20} {"Price"}')
print('-' * 95)
for row in results3:
    print(f'{str(row[0]):<65} {str(row[1]):<20} £{row[2]}')

## Step 7: Close Connection

In [ ]:
# Always close the cursor and connection when finished
# This releases database resources and prevents connection leaks
cursor.close()
conn.close()
print('MySQL connection closed.')